# Crivo: preservar fatos após uma hipótese

Carrega o transformer próprio anterior e acrescenta memória explícita de fatos/alternativas. Os dois novos treinos foram rejeitados por regressões. Esta melhoria é de um sistema híbrido, com regras de memória e cópia literal; não é aprendizado de conversa livre. Não precisa iniciar treino ou contratar GPU para experimentar.


**Experimental: não passou o critério geral.** Nos368 casos inéditos,133→143 acertos; cobertura da memória11/368. O exemplo guiado foi corrigido, mas isso não demonstra conversa livre. Os erros e os casos em que a memória se absteve estão publicados.


In [ ]:
from pathlib import Path
import subprocess,sys,json
REPO=Path('/content/CRIVO-memoria-confirmada')
BRANCH='codex/memoria-fontes-20261009'
if not REPO.exists():
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/HROSONE/CRIVO.git',str(REPO)])
else:
    if subprocess.check_output(['git','branch','--show-current'],cwd=REPO,text=True).strip()!=BRANCH:
        raise RuntimeError('Pasta de outra branch; escolha uma pasta nova.')
print('Revisão:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
LAB=REPO/'experimentos/memoria_explicita_20261009'
FONTES=REPO/'experimentos/memoria_fontes_20261009'
if not (LAB/'avaliacao/resumo.json').exists():
    raise RuntimeError('Checkout antigo ou incompleto. Recrie a pasta em uma sessão nova para baixar a versão publicada.')


In [ ]:
subprocess.check_call([sys.executable,'-m','pip','install','torch>=2.2,<3','numpy>=1.24,<3','tokenizers>=0.20,<1'])


## Resultados e limites

O painel novo foi reservado até congelar o código. Outro agente escreveu os casos sem ler o ledger ou suas previsões. Continua sendo gramática finita e autoria da mesma equipe; não é teste independente externo. Os demais painéis já eram conhecidos. Os traços incluem erros e abstenções, com origem literal dos argumentos.


Com os mesmos pesos, o painel conhecido de retenção melhora230→280/370; o novo painel melhora só36,14%→38,86%, uma conversa inteira correta em72. Os outros painéis conhecidos e as regressões estão na tabela impressa abaixo.


In [ ]:
r=json.loads((LAB/'avaliacao/resumo.json').read_text())
for painel,condicoes in r['resultados'].items():
    for modo,m in condicoes.items():
        print(painel,modo,m['contratos_corretos'],'/',m['n'],'sessões completas',m['sessoes_completas'],'/',m['sessoes'])
        if modo=='memoria_explicita':print('Cobertura',m['memoria_aplicada'],'abstenções',m['abstencoes'])
print('Critérios:',r['criterios'])
print('Limites:',r['limites'])


## Teste suas falas

`USAR_MEMORIA=True` compõe o modelo anterior com memória explícita. `False` mostra só a proposta neural, com o mesmo preparo novo. Não apresenta a cabeça auxiliar de evento antiga como classificação válida.

As fontes apontam para trechos da conversa. Operações fora do suporte ou situações ambíguas preservam a proposta neural e mostram a abstenção. Isso não torna a proposta neural correta. Nomes de uma palavra, gramática restrita e no máximo256 tokens, sem truncamento. O arquivo temporário fica na sessão, sem envio automático ao Drive ou treino.


Na inspeção local com o modelo real, o exemplo abaixo passa5/5 com memória, contra4/5 sem memória. Após confirmar19 reais, o retorno mantém19; sem memória, copia41. Essa sequência é um teste de comportamento conhecido, separado dos368 casos inéditos.


In [ ]:
USAR_MEMORIA=True
FALAS=[
    'O plano Azul custa 23 reais. O plano Verde custa 35 reais. Compare Azul e Verde.',
    'Corrigindo Azul: o custo real é 41 reais. Compare Azul e Verde.',
    'Imagine Azul por 19 reais, sem alterar os fatos. Compare Azul e Verde.',
    'A última hipótese aconteceu de verdade. Agora ela é fato. Compare Azul e Verde.',
    'Esqueça a simulação e use os preços reais corrigidos. Compare Azul e Verde.',
]
p=Path('/content/falas-memoria-confirmada.json');p.write_text(json.dumps(FALAS,ensure_ascii=False))
RUNNER="import sys,json,torch\nfrom pathlib import Path\nsys.path.insert(0,sys.argv[1]);sys.path.insert(0,sys.argv[2])\nfrom apoio import ROOT,INICIAL\nfrom rede_eventos import carregar\nfrom preparo import codificar\nfrom modelo import lote,executar\nfrom decodificar import proposta_limitada\nfrom ledger import aplicar\nfrom tokenizers import Tokenizer\ntorch.set_num_threads(1)\ntok=Tokenizer.from_file(str(ROOT/'artefatos/linguagem_profunda/tokenizer.json'));tok.encode_special_tokens=True\nm,_=carregar(path=INICIAL);m.eval()\nfalas=json.loads(Path(sys.argv[3]).read_text())\nfor i in range(1,len(falas)+1):\n    it=codificar(tok,{'turnos':falas[:i],'argumentos':[None]*3,'referente':None})\n    x,l=lote([it],tok.token_to_id('<pad>'))\n    with torch.no_grad():ls=m(x,l)\n    neural=proposta_limitada(it,ls,0)\n    if sys.argv[4]=='True':pr,diag=aplicar(falas[:i],neural)\n    else:pr,diag=neural,{'aplicado':False,'motivo':'Memória desligada pelo usuário.'}\n    print(json.dumps({'turno':i,'memoria':diag,'proposta_neural':neural,'proposta':pr,'execucao':executar(pr)},ensure_ascii=False))\n"
subprocess.check_call([sys.executable,'-c',RUNNER,str(LAB),str(FONTES),str(p),str(USAR_MEMORIA)])


## O que foi treinado

A rodada anterior fez dois treinos de400 passos ajustando135.692 parâmetros próprios. Foram concluídos e avaliados, mas os candidatos regrediram e não são usados nesta composição. Todos os checkpoints e relatórios continuam em `experimentos/memoria_fontes_20261009`.

A memória explícita usa o checkpoint anterior de retenção, sem modificar seus pesos. O transformer tem aproximadamente2,95 milhões de parâmetros nesta interface; o ledger não possui parâmetros treináveis. Não há modelos, tokenizadores ou APIs externos. Nenhum peso ativo do chat é substituído.

Os scripts permitem reproduzir avaliação e inspecionar resultados. O caderno é verificado localmente em CPU; não foi executado numa sessão real do Colab.
